##### Import statements:

In [ ]:
import os
import pathlib
import pandas as pd
import numpy as np
import pickle
from analysis_metadata.analysis_metadata import Metadata, increment_dir_name, write_metadata
import socket

##### Define input paths and parameters:

In [ ]:
# Define input paths:
input_paths = [
    os.path.join('/', 'mnt', 'smb', 'locker', 'issa-locker', 'users', 'Dan', 'code', 'ws', 'results', 'run780', 'ae_iterate_beta_reconstruction.pickle'),
    os.path.join('/', 'mnt', 'smb', 'locker', 'issa-locker', 'users', 'Dan', 'code', 'ws', 'results', 'run781', 'ae_iterate_beta_reconstruction.pickle')
    ]
    
# Define output settings:
save_output = True
hostname = socket.gethostname()
if 'rc.zi.columbia.edu' in hostname:
    base_output_directory = os.path.join('/', 'mnt', 'smb', 'locker', 'issa-locker', 'users', 'Dan', 'code', 'ws', 'results')
elif hostname == 'DESKTOP-1PVCRAF':
    base_output_directory='E:\\simulation_whiskers\\results\\'
folder_basename = 'run'
file_basename = 'ae_iterate_beta_reconstruction'

##### Merge together input dataframes: 

In [ ]:
geo_df = pd.DataFrame()
perf_df = pd.DataFrame()
ae_df = pd.DataFrame()

for inpt in input_paths:
    
    # Load:
    curr_result = pickle.load(open(inpt,'rb'))
    curr_geo_df = curr_result['geo_df']
    curr_perf_df = curr_result['perf_df']
    curr_ae_df = curr_result['ae_df']
    
    # Append source file:
    curr_geo_df['src'] = [inpt]*curr_geo_df.shape[0]
    curr_perf_df['src'] = [inpt]*curr_perf_df.shape[0]
    curr_ae_df['src'] = [inpt]*curr_ae_df.shape[0]
    
    # Merge:
    geo_df = pd.concat([geo_df, curr_geo_df])
    perf_df = pd.concat([perf_df, curr_perf_df])
    ae_df = pd.concat([ae_df, curr_ae_df])

# Re-number repeats (to avoid collisions between runs):
abs_rep_df = ae_df[['src', 'repeat_idx']].drop_duplicates().reset_index()
abs_rep_df['repeat_abs'] = np.arange(abs_rep_df.shape[0])

geo_df = pd.merge(geo_df, abs_rep_df, on=['src', 'repeat_idx'], how='outer')
perf_df = pd.merge(perf_df, abs_rep_df, on=['src', 'repeat_idx'], how='outer')
ae_df = pd.merge(ae_df, abs_rep_df, on=['src', 'repeat_idx'], how='outer')

# Rename columns:
geo_df = geo_df.rename(columns={'repeat_idx':'repeat_within_run', 'repeat_abs':'repeat'})
perf_df = perf_df.rename(columns={'repeat_idx':'repeat_within_run', 'repeat_abs':'repeat'})
ae_df = ae_df.rename(columns={'repeat_idx':'repeat_within_run', 'repeat_abs':'repeat'})

results = dict()
results['geo_df'] = geo_df
results['perf_df'] = perf_df
results['ae_df'] = ae_df

##### Save results:

In [ ]:
if save_output: 
    
    # Save results dataframe:
    curr_output_directory=increment_dir_name(base_output_directory, folder_basename)
    if not os.path.exists(curr_output_directory):
        pathlib.Path(curr_output_directory).mkdir(parents=True, exist_ok=True)
    results_path = os.path.join(curr_output_directory, file_basename+'.pickle')
    pickle.dump(results, open(results_path, 'wb'))
    
    # Save metadata:
    M = Metadata()
    for inpt in input_paths:
        M.add_input(inpt)
    metadata_path = os.path.join(curr_output_directory, 'stitch_ae_results_metadata.json')
    write_metadata(M, metadata_path)